# Task vs progress: what does the pivot read the goal from?

The KV-transfer results argue for **repeated re-reading** of the goal from the task span, against a **sentinel** view
(a goal marker built up in the progress tokens and carried forward). This notebook fills the 2 × 2 grid of *which span the
pivot reads from the source run*, on the multi-predicate setup:

```
p_src: a'b'c -> c     source (donor)          p_dst: a'b'd -> d     the patched run (prompt ends at "{var} =")
```

| | progress from DST | progress from SRC |
|---|---|---|
| **task from DST** | no-op (clean DST) | pre-commitment swap: 6 % (residual, all layers) |
| **task from SRC** | task-span KV, goal-250: 37 % (V×1), 80 % (V×4) | ≈ 99 % |

Spans: **task** = the task list in the user turn (`\n`-padded so SRC/DST align); **progress** = `<|im_end|> … {var}` (the
prefilled code; token-identical in SRC and DST). The pivot ` =` is never overwritten; everything is applied **from the
pivot onward**.

**Methods**
* **`kv`** (`PP.intervention_stack`, I3 only): the chosen heads (`goal250` or `all`) read SRC's k/v on the span from the
  pivot on; values × `V`. Every other key, and every pre-pivot latent, stays as in the clean DST run.
* **`residual`**: every layer's output residual on the span ← SRC's; every *other* position between the span and the
  pivot ← clean DST's, so it stays identical to the clean DST run (e.g. task swapped ⇒ the progress is pinned to DST).

At V×1, `kv` with **all** heads and `residual` give the pivot the same keys and values, so they should agree (checked
below). The open question: does boosting the *progress* values (V×4) raise the 6 %? That would mean the progress carries
a pre-commitment that contributes, though less than the task span.

Same instance sampler as the batch runs (`PP.sample_multi_instances`, seed 1123): instance *i* here is instance *i* in
`path_patch_eval.py` / `precommit_swap_eval.py`. Python → Python for now.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")   # check nvidia-smi first (shared machine)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import json, collections
from contextlib import ExitStack
from types import SimpleNamespace
import numpy as np
import torch
import src.diff.functional as P
import src.diff.functional as FUNC
from src.diff.functional import *          # helpers into scope
import src.diff.path_patch as PP            # geometry / k-v routing / residual tracer + patcher / oracle (batch eval's code)
import src.diff.task_vs_progress as TP      # this notebook's grid: geometry / interventions / resumable sweep (shared with the batch driver)

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
with open(os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR),
                       "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")) as f:
    GOAL_HEADS_250 = [(l, h) for l, h in json.load(f)]
ALL_HEADS = [(l, h) for l in range(mt.n_layer) for h in range(mt.n_heads)]
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich.console import Console
from rich.markup import escape
print(f"{len(GOAL_HEADS_250)} goal heads · {len(ALL_HEADS)} heads in total")

2026-10-26 13:51:15 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228
250 goal heads · 2560 heads in total


## Knobs

In [2]:
#------------------------------------------------------------------------------------------------------
DATATYPE = "list[str]"            # int | str | list[int] | list[str]
SRC_LANG, DST_LANG = "python", "python"
SEED, N_SAMPLES = 1123, 32        # PP.sample_multi_instances (the batch eval's draw)
IDX = 0                           # instance for the single-instance cells
READOUT_HEADS = GOAL_HEADS_250
READOUT = True                    # GoalScope readout at ` =` in the single-instance run (the sweep has its own knob)
MAX_NEW = 64
ORACLE_MODEL = "claude-sonnet-4-6"
HEADSETS = {"goal250": GOAL_HEADS_250, "all": ALL_HEADS}
CONFIGS = list(TP.CONFIGS)        # (method, heads, span, V): none + kv {goal250, all} x {task, progress, task+progress} x V {1, 2, 4} + residual x 3
#------------------------------------------------------------------------------------------------------
PP_CTX = SimpleNamespace(mt=mt, group=P.GROUP, readout_heads=READOUT_HEADS, gsp=getattr(P, "gsp", None))
LAYERS = list(range(mt.n_layer))
cfg_name = TP.cfg_name

## Geometry — task span, progress span, pivot

`PP.build_geometry(window="span", span="list")` gives the `\n`-padded task-list windows and the I2 progress window
(`<|im_end|> … pivot-1` on DST); the SRC progress positions are the same tokens aligned from the end (asserted). Tokens
between the task list and the progress (the rest of the user turn), if any, are shown in grey (task span: light red in SRC, light green in DST; progress: yellow; pivot: salmon): they are never transferred
(`kv`: read from DST; `residual`: pinned to DST).

In [3]:
def geometry(inst):   # task = `(1)` .. <|im_end|> (excluded; PP.build_geometry span="list+gap"), progress = <|im_end|> .. {var}
    return TP.geometry(inst, ctx=PP_CTX)


TASK_STYLE = {"src": "black on #ffb3b3", "dst": "black on #b8f0b8"}   # source task span: light red · destination: light green


def show_prompt(G, which="dst"):
    tok = G[f"{which}_tok"]; ids = PP._ids(tok)
    task = set(G["W"]["task"][0 if which == "src" else 1]); prog = set(G["W"]["progress"][0 if which == "src" else 1])
    gap = set(G["gap"]) if which == "dst" else set()
    piv = len(ids) - 1
    t = Text()
    for i, tid in enumerate(ids):
        s = mt.tokenizer.decode([tid]).replace("\n", "\\n\n")
        style = ("bold black on salmon1" if i == piv else TASK_STYLE[which] if i in task
                 else "black on yellow1" if i in prog else "grey50" if i in gap else "")
        t.append(s, style=style)
    tcol = "light red" if which == "src" else "light green"
    rich_print(Panel(t, title=f"[bold]{which.upper()}[/bold] · task span ({tcol}) · progress (yellow) · pivot (salmon) · "
                              f"L={len(ids)}", title_align="left", border_style="grey42"))


INSTS, _sample_seed = PP.sample_multi_instances(DATATYPE, SRC_LANG, DST_LANG, n_samples=N_SAMPLES, seed=SEED)
G = geometry(INSTS[IDX])
inst = G["inst"]
print(f"[{IDX}] SRC c = {inst.src_frontier.id}: {inst.src_frontier.description!r}\n"
      f"     DST d = {inst.dst_frontier.id}: {inst.dst_frontier.description!r}")
print(f"task {len(G['W']['task'][1])} tok (incl. {G['n_gap']} task-to-progress tok) · progress {len(G['W']['progress'][1])} tok "
      f"· pad {G['g']['pad_src']}/{G['g']['pad_dst']} (src/dst) · pivot {mt.tokenizer.decode([PP._ids(G['dst_tok'])[-1]])!r}")
show_prompt(G, "src"); show_prompt(G, "dst")

[0] SRC c = count_long: 'the number of words longer than five characters'
     DST d = longest_word_length: 'the length of the longest word'
task 35 tok (incl. 1 task-to-progress tok) · progress 56 tok · pad 0/2 (src/dst) · pivot ' ='


╭─ SRC · task span (light red) · progress (yellow) · pivot (salmon) · L=155 ──────────────────────────────────────╮
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words that start with a vowel\n                                                               │
│ (2) whether any word is a palindrome\n                                                                          │
│ (3) the number of words longer than five characters\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w and w[0].lower() in "aeiou")\n                                                │
│     b = any(w == w[::-1] for w in words)\n                                                                      │
│     c =                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · task span (light green) · progress (yellow) · pivot (salmon) · L=155 ────────────────────────────────────╮
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ (1) the number of words that start with a vowel\n                                                               │
│ (2) whether any word is a palindrome\n                                                                          │
│ (3) the length of the longest word\n                                                                            │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w and w[0].lower() in "aeiou")\n                                                │
│     b = any(w == w[::-1] for w in words)\n                                                                      │
│     c =                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Interventions

`make_stack(G, method, span, heads, V)` → a zero-argument factory of a fresh context (single use per forward /
generation), so every config runs through the same `run_patched`.

`residual` also swaps the span's **input embeddings** (layer 0 reads k/v from the embeddings, not from a layer output), so at every layer the pivot sees exactly SRC's span: this is what makes it equal to `kv` with all heads at V×1.

In [4]:
def make_stack(G, method, span, heads=None, v_scale=1.0):
    return TP.make_stack(G, method, span, heads, v_scale, headsets=HEADSETS, layers=LAYERS, ctx=PP_CTX)


def run_patched(G, stack, readout=False):
    return TP.run_patched(G, stack, max_new=MAX_NEW, readout_heads=READOUT_HEADS if readout else None, ctx=PP_CTX)


def judge(G, clean, gen, ro):
    return TP.judge(G, clean, gen, ro, model_name=ORACLE_MODEL)


def pivot_logits(G, stack):
    return TP.pivot_logits(G, stack, ctx=PP_CTX)


## Sanity checks

1. **Pre-pivot latents.** `kv`: every position before the pivot must equal the clean DST run (the transfer acts from the
   pivot on). `residual`: the span equals SRC's residuals, every other pre-pivot position equals clean DST.
2. **`kv` (all heads, V×1) ≡ `residual`** at the pivot, per span: max |Δ logit| and the top-1 token (bf16, so expect
   small but nonzero differences).

In [5]:
pre = list(range(G["piv_d"]))
h_dst = PP.layer_outputs(mt, G["dst_tok"], pre, LAYERS)
t = Table(title="1 · pre-pivot latents vs clean DST (max |Δ| over all layers)")
for c in ("config", "non-span positions", "span positions vs SRC"):
    t.add_column(c)
for m, h, s, v in [("kv", "goal250", "task+progress", 4.0), ("kv", "all", "task+progress", 4.0),
                   ("residual", None, "task", 1.0), ("residual", None, "progress", 1.0), ("residual", None, "task+progress", 1.0)]:
    st = make_stack(G, m, s, h, v)
    with st():
        h_pat = PP.layer_outputs(mt, G["dst_tok"], pre, LAYERS)
    span_d = set(G["W"][s][1])
    keep = [i for i, q in enumerate(pre) if m == "kv" or q not in span_d]
    d_keep = max(float((h_pat[l][keep] - h_dst[l][keep]).abs().max()) for l in LAYERS)
    if m == "residual":
        idx = [pre.index(q) for q in G["W"][s][1]]
        h_src = PP.layer_outputs(mt, G["src_tok"], G["W"][s][0], LAYERS)
        d_span = max(float((h_pat[l][idx] - h_src[l]).abs().max()) for l in LAYERS)
    else:
        d_span = float("nan")
    t.add_row(cfg_name(m, h, s, v), f"{d_keep:.3g}  {'OK' if d_keep == 0 else 'CHANGED'}",
              "—" if m == "kv" else f"{d_span:.3g}  {'OK' if d_span < 1e-2 else 'MISMATCH'}")
Console(width=140).print(t)

t = Table(title="2 · kv (all heads, V×1) vs residual at the pivot")
for c in ("span", "max |Δ logit|", "top-1 kv", "top-1 residual", "top-1 clean DST"):
    t.add_column(c)
top = lambda lg: repr(mt.tokenizer.decode([int(lg.argmax())]))
lg_dst = pivot_logits(G, ExitStack)
for s in ("task", "progress", "task+progress"):
    a, b = pivot_logits(G, make_stack(G, "kv", s, "all", 1.0)), pivot_logits(G, make_stack(G, "residual", s))
    t.add_row(s, f"{float((a - b).abs().max()):.3f}", top(a), top(b), top(lg_dst))
Console(width=140).print(t)

          1 · pre-pivot latents vs clean DST (max |Δ| over all layers)           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ config                           ┃ non-span positions ┃ span positions vs SRC ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ kv·goal250 · task+progress · V×4 │ 0  OK              │ —                     │
│ kv·all · task+progress · V×4     │ 0  OK              │ —                     │
│ residual · task · V×1            │ 0  OK              │ 0  OK                 │
│ residual · progress · V×1        │ 0  OK              │ 0  OK                 │
│ residual · task+progress · V×1   │ 0  OK              │ 0  OK                 │
└──────────────────────────────────┴────────────────────┴───────────────────────┘

               2 · kv (all heads, V×1) vs residual at the pivot                
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ span          ┃ max |Δ logit| ┃ top-1 kv ┃ top-1 residual ┃ top-1 clean DST ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ task          │ 0.000         │ ' sum'   │ ' sum'         │ ' max'          │
│ progress      │ 0.000         │ ' max'   │ ' max'         │ ' max'          │
│ task+progress │ 0.000         │ ' sum'   │ ' sum'         │ ' max'          │
└───────────────┴───────────────┴──────────┴────────────────┴─────────────────┘

## One instance — every config

Clean SRC / DST once (with readouts if `READOUT`), then each config in `CONFIGS`: patched generation, GoalScope at ` =`,
and the oracle verdict (`PP.judge`: c = the source frontier, BOTH is a miss).

In [6]:
clean = PP.clean_runs(dict(src_tok=G["src_tok"], dst_tok=G["dst_tok"]), readout_heads=READOUT_HEADS, max_new=MAX_NEW,
                      readout=READOUT, ctx=PP_CTX)
rich_print(Panel(Text(clean["gen_src"].strip()), title=f"[red1]clean SRC (c = {inst.src_frontier.id})[/red1]"
                 + (f" · readout: {escape(clean['ro_src'])}" if READOUT else ""), title_align="left"))
rich_print(Panel(Text(clean["gen_dst"].strip()), title=f"[sea_green3]clean DST (d = {inst.dst_frontier.id})[/sea_green3]"
                 + (f" · readout: {escape(clean['ro_dst'])}" if READOUT else ""), title_align="left"))
ONE = []
for m, h, s, v in CONFIGS:
    gen, ro = (clean["gen_dst"], clean.get("ro_dst")) if m == "none" else run_patched(G, make_stack(G, m, s, h, v), READOUT)
    causal, readout = judge(G, clean, gen, ro)
    ONE.append(dict(cfg=cfg_name(m, h, s, v), causal=causal["label"], readout=readout["patched"] if readout else None,
                    gen=gen, ro=ro))
    first = next((ln for ln in gen.split("\n") if ln.strip()), "").strip()
    rich_print(f"[bold]{escape(ONE[-1]['cfg']):<34}[/bold] causal=[bold]{causal['label']:<8}[/bold]"
               + (f" readout={readout['patched']:<8}" if readout else "") + f" · {escape(first[:90])}")

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


pos=154 ' =' => I'm counting lines where the length of the string is greater than 5.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=154 ' =' => Identifying the longest string length in a list.

╭─ clean SRC (c = count_long) · readout: I'm counting lines where the length of the string is greater than 5. ────╮
│ sum(1 for w in words if len(w) > 5)                                                                             │
│     return a, b, c                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (37 outputs left out)

residual · progress · V×1          causal=dst      readout=dst      · max((len(w) for w in words), default=0)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=154 ' =' => I'm counting lines where the length of the string is greater than 5.

residual · task+progress · V×1     causal=src      readout=src      · sum(1 for w in words if len(w) > 5)

## Sweep — the 2 × 2 grid over instances

Clean runs once per instance (no readouts unless `SWEEP_READOUT`), then every config. Rows are stored in `SWEEP`
(keyed by the config name, so re-running with more instances / configs adds to it).

**Resume** (`RESUME = True`): each instance's clean runs and per-config results are cached in `results/notebook_caches/task_vs_progress/<model>/<datatype>__<src>-<dst>__seed…__mx<MAX_NEW>__ro<0|1>/`, one JSON per instance, written after every config. Re-running skips cached (instance, config) pairs; new configs in `CONFIGS` or new instances run fresh. Delete the folder (or set `RESUME = False`) to recompute.

In [7]:
SWEEP_IDX = range(32)          # instances of DATATYPE (N_SAMPLES drawn)
SWEEP_READOUT = False
RESUME = True                  # reuse cached clean runs + config results per instance (results/notebook_caches/...)
CACHE_DIR = TP.cache_dir(env_utils.DEFAULT_RESULTS_DIR, mt.name.split("/")[-1], DATATYPE, SRC_LANG, DST_LANG,
                         seed=SEED, max_new=MAX_NEW, readout=SWEEP_READOUT)
os.makedirs(CACHE_DIR, exist_ok=True)

SWEEP = globals().get("SWEEP", collections.defaultdict(list))
n_run = n_hit = 0
for i in SWEEP_IDX:
    rows, r_, h_ = TP.sweep_instance(INSTS[i], i, CONFIGS, CACHE_DIR, headsets=HEADSETS, readout_heads=READOUT_HEADS,
                                     readout=SWEEP_READOUT, max_new=MAX_NEW, oracle_model=ORACLE_MODEL, resume=RESUME,
                                     layers=LAYERS, ctx=PP_CTX)
    n_run, n_hit = n_run + r_, n_hit + h_
    for name, row in rows.items():
        key = (DATATYPE, name)
        SWEEP[key] = [r for r in SWEEP[key] if r["idx"] != i] + [row]
    line = ["■" if rows[cfg_name(*c)]["causal"] == "src" else "·" for c in CONFIGS]
    print(f"[{i:2d}] {INSTS[i].src_frontier.id} vs {INSTS[i].dst_frontier.id}: {' '.join(line)}")
print(f"configs run: {n_run} · from cache: {n_hit} · cache: {CACHE_DIR}")


[ 0] count_long vs longest_word_length: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[ 1] shortest_word_length vs count_long: · ■ ■ · ■ ■ ■ ■ · ■ ■ ■ · ■
[ 2] total_uppercase_letters vs count_with_digit: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[ 3] count_short vs total_vowels: · · ■ · · ■ ■ ■ · · ■ ■ · ■
[ 4] count_long vs count_all_lowercase: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[ 5] distinct_count vs all_unique_lengths: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[ 6] distinct_count vs longest_word_length: · · ■ · · ■ ■ ■ · ■ ■ ■ · ■
[ 7] total_uppercase_letters vs all_unique_lengths: · · ■ · · ■ ■ ■ · · ■ ■ · ■
[ 8] count_short vs count_ends_consonant: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[ 9] count_starts_vowel vs count_capitalized: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[10] total_length vs total_uppercase_letters: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[11] has_palindrome vs distinct_count: · ■ ■ · ■ ■ ■ ■ ■ ■ ■ ■ ■ ■
[12] longest_word_length vs count_all_lowercase: · · ■ · · ■ · ■ · ■ ■ · · ■
[13] count_with_digit vs total_uppercase_letters: · ■ ■ · · ■ ■ ■ · · 

## Results — 2 × 2 grids and the full config table

Causal-shift (patched writes c) per config; each 2 × 2 grid fixes the method / heads / V and varies which span the pivot
reads from SRC (the no-op corner is the clean DST run).

In [8]:
def rate(rs, f="shifted"):
    xs = [r[f] for r in rs if r[f] is not None]
    return (sum(xs) / len(xs)) if xs else float("nan")

for dt in dict.fromkeys(k[0] for k in SWEEP):
    t = Table(title=f"{escape(dt)} · all configs", show_lines=False)
    for c, j in (("config", "left"), ("n", "right"), ("causal-shift", "right"), ("readout-shift", "right"), ("verdicts", "left")):
        t.add_column(c, justify=j)
    for (d, name), rs in SWEEP.items():
        if d != dt:
            continue
        t.add_row(name, str(len(rs)), f"{rate(rs):.2f}", f"{rate(rs, 'ro_shifted'):.2f}",
                  " ".join(f"{k}:{n}" for k, n in collections.Counter(r["causal"] for r in rs).most_common()))
    Console(width=140).print(t)

    noop = SWEEP.get((dt, "clean DST (no-op)"), [])
    groups = dict.fromkeys((r["method"], r["heads"], r["v"]) for (d, _), rs in SWEEP.items() if d == dt for r in rs[:1]
                           if r["method"] != "none")
    for m, h, v in groups:
        cell = {}
        for s in ("task", "progress", "task+progress"):
            cell[s] = SWEEP.get((dt, cfg_name(m, h, s, v)))
        g2 = Table(title=f"{escape(dt)} · {m}{'·' + h if h else ''} · V×{v:g} — causal-shift", show_lines=True)
        g2.add_column(""); g2.add_column("progress from DST", justify="center"); g2.add_column("progress from SRC", justify="center")
        f = lambda rs: f"{rate(rs):.2f} (n={len(rs)})" if rs else "—"
        g2.add_row("task from DST", f(noop), f(cell["progress"]))
        g2.add_row("task from SRC", f(cell["task"]), f(cell["task+progress"]))
        Console(width=100).print(g2)

                                        list[str] · all configs                                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ config                           ┃  n ┃ causal-shift ┃ readout-shift ┃ verdicts                      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ clean DST (no-op)                │ 32 │         0.00 │           nan │ dst:32                        │
│ kv·goal250 · task · V×1          │ 32 │         0.78 │           nan │ src:25 dst:4 neither:3        │
│ kv·goal250 · task · V×4          │ 32 │         1.00 │           nan │ src:32                        │
│ kv·goal250 · progress · V×1      │ 32 │         0.00 │           nan │ dst:32                        │
│ kv·goal250 · progress · V×4      │ 32 │         0.16 │           nan │ dst:22 src:5 neither:4 both:1 │
│ kv·goal250 · task+progress · V×1 │ 32 │         1.00 │           nan │ src:32                        │
│ kv·all · task · V×1              │ 32 │         0.94 │           nan │ src:30 dst:2                  │
│ kv·all · task · V×4              │ 32 │         0.97 │           nan │ src:31 neither:1              │
│ kv·all · progress · V×1          │ 32 │         0.03 │           nan │ dst:31 src:1                  │
│ kv·all · progress · V×4          │ 32 │         0.31 │           nan │ dst:21 src:10 neither:1       │
│ kv·all · task+progress · V×1     │ 32 │         1.00 │           nan │ src:32                        │
│ residual · task · V×1            │ 32 │         0.94 │           nan │ src:30 dst:2                  │
│ residual · progress · V×1        │ 32 │         0.03 │           nan │ dst:31 src:1                  │
│ residual · task+progress · V×1   │ 32 │         1.00 │           nan │ src:32                        │
└──────────────────────────────────┴────┴──────────────┴───────────────┴───────────────────────────────┘

       list[str] · kv·goal250 · V×1 — causal-shift       
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃               ┃ progress from DST ┃ progress from SRC ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ task from DST │    0.00 (n=32)    │    0.00 (n=32)    │
├───────────────┼───────────────────┼───────────────────┤
│ task from SRC │    0.78 (n=32)    │    1.00 (n=32)    │
└───────────────┴───────────────────┴───────────────────┘

       list[str] · kv·goal250 · V×4 — causal-shift       
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃               ┃ progress from DST ┃ progress from SRC ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ task from DST │    0.00 (n=32)    │    0.16 (n=32)    │
├───────────────┼───────────────────┼───────────────────┤
│ task from SRC │    1.00 (n=32)    │         —         │
└───────────────┴───────────────────┴───────────────────┘

         list[str] · kv·all · V×1 — causal-shift         
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃               ┃ progress from DST ┃ progress from SRC ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ task from DST │    0.00 (n=32)    │    0.03 (n=32)    │
├───────────────┼───────────────────┼───────────────────┤
│ task from SRC │    0.94 (n=32)    │    1.00 (n=32)    │
└───────────────┴───────────────────┴───────────────────┘

         list[str] · kv·all · V×4 — causal-shift         
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃               ┃ progress from DST ┃ progress from SRC ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ task from DST │    0.00 (n=32)    │    0.31 (n=32)    │
├───────────────┼───────────────────┼───────────────────┤
│ task from SRC │    0.97 (n=32)    │         —         │
└───────────────┴───────────────────┴───────────────────┘

        list[str] · residual · V×1 — causal-shift        
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃               ┃ progress from DST ┃ progress from SRC ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ task from DST │    0.00 (n=32)    │    0.03 (n=32)    │
├───────────────┼───────────────────┼───────────────────┤
│ task from SRC │    0.94 (n=32)    │    1.00 (n=32)    │
└───────────────┴───────────────────┴───────────────────┘

## Inspect cases — filter by config and outcome, then re-run one

**Filter** (on `SWEEP`): which spans the pivot reads from SRC (`FILTER_SPAN`), the scheme (`FILTER_SCHEME`: `residual` ·
`kv·goal250` · `kv·all` · `none`), its V-boost (`FILTER_V`), and the outcome (`FILTER_CRITERION`: `success` = the patched
run implements c, `fail` = anything else, `any`). **Re-run** the `PICK`-th match under the same intervention, with the
V-boost overridable for `kv` (`INSPECT_V`): clean SRC / DST and patched generations, GoalScope readouts at ` =`, and the
oracle verdict, next to the swept verdict.

The re-run is split in two: the **clean** cell (`PICK`) shows both prompts with the spans highlighted and runs clean SRC / DST; the **patched** cell has the intervention knobs (`INSPECT_SPAN`, `INSPECT_SCHEME`, `INSPECT_V`; default = the filter's) and first checks the latents before the pivot: every span that is **not** patched must equal the clean DST run, the patched span must equal SRC (`residual`) or stay DST (`kv`, which only changes what the pivot onward reads).

In [20]:
#------------------------------------------------------------------------------------------------------
FILTER_SPAN = "progress"        # "task" | "progress" | "task+progress" | "none"
FILTER_SCHEME = "kv·goal250"        # "residual" | "kv·goal250" | "kv·all" | "none" (clean DST)
FILTER_V = 4.0                  # V-boost of the swept config to filter on (residual / none: 1.0)
FILTER_CRITERION = "success"    # "success" (patched = c) | "fail" (dst / both / neither) | "any"
#------------------------------------------------------------------------------------------------------
SCHEMES = {"residual": ("residual", None), "kv·goal250": ("kv", "goal250"), "kv·all": ("kv", "all"), "none": ("none", None)}
F_METHOD, F_HEADS = SCHEMES[FILTER_SCHEME]
F_NAME = cfg_name(F_METHOD, F_HEADS, "none" if F_METHOD == "none" else FILTER_SPAN, 1.0 if F_METHOD != "kv" else FILTER_V)
rows = SWEEP.get((DATATYPE, F_NAME), [])
assert rows, f"no swept results for {F_NAME!r} ({DATATYPE}) — run the sweep with this config in CONFIGS"
keep = {"success": lambda r: r["shifted"], "fail": lambda r: not r["shifted"], "any": lambda r: True}[FILTER_CRITERION]
MATCHES = sorted([r for r in rows if keep(r)], key=lambda r: r["idx"])

t = Table(title=f"{escape(DATATYPE)} · {F_NAME} · {FILTER_CRITERION}: {len(MATCHES)}/{len(rows)}", show_lines=False)
for c in ("#", "idx", "c (SRC)", "d (DST)", "verdict", "patched first line"):
    t.add_column(c)
for k, r in enumerate(MATCHES):
    first = next((ln for ln in r["gen"].split("\n") if ln.strip()), "").strip()
    t.add_row(str(k), str(r["idx"]), r["c"], r["d"], Text(r["causal"], style="green4" if r["shifted"] else "grey50"), first[:70])
Console(width=160).print(t)

                                   list[str] · kv·goal250 · progress · V×4 · success: 5/32                                    
┏━━━┳━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ idx ┃ c (SRC)              ┃ d (DST)                 ┃ verdict ┃ patched first line                                    ┃
┡━━━╇━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0 │ 1   │ shortest_word_length │ count_long              │ src     │ min((len(w) for w in words if len(w) > 5), default=0) │
│ 1 │ 11  │ has_palindrome       │ distinct_count          │ src     │ any(w == w[::-1].capitalize() for w in words)         │
│ 2 │ 15  │ total_vowels         │ longest_word_length     │ src     │ sum(w.count(v) for w in words for v in 'aeiouAEIOU')  │
│ 3 │ 19  │ longest_word_length  │ count_capitalized       │ src     │ max(len(w) for w in words, default=0)                 │
│ 4 │ 25  │ shortest_word_length │ total_uppercase_letters │ src     │ min(len(w) for w in words if w) if words else 0       │
└───┴─────┴──────────────────────┴─────────────────────────┴─────────┴───────────────────────────────────────────────────────┘

In [21]:
# ── the picked case: prompts (spans highlighted) + clean SRC / DST runs ──
#------------------------------------------------------------------------------------------------------
PICK = 2                        # which row of the filtered table (column "#")
INSPECT_READOUT = True          # GoalScope readouts at ` =` for clean SRC / DST (and the patched run below)
#------------------------------------------------------------------------------------------------------
assert MATCHES, "no case matches the filter"
r0 = MATCHES[PICK]
Gx = geometry(INSTS[r0["idx"]])
ix = Gx["inst"]
rich_print(f"[bold]idx {r0['idx']}[/bold] · c = {ix.src_frontier.id}: {escape(ix.src_frontier.description)}\n"
           f"         d = {ix.dst_frontier.id}: {escape(ix.dst_frontier.description)}\n"
           f"task {len(Gx['W']['task'][1])} tok (incl. {Gx['n_gap']} task-to-progress tok) · progress {len(Gx['W']['progress'][1])} tok · "
           f"pad {Gx['g']['pad_src']}/{Gx['g']['pad_dst']} (src/dst)")
show_prompt(Gx, "src"); show_prompt(Gx, "dst")
cx = PP.clean_runs(dict(src_tok=Gx["src_tok"], dst_tok=Gx["dst_tok"]), readout_heads=READOUT_HEADS, max_new=MAX_NEW,
                   readout=INSPECT_READOUT, ctx=PP_CTX)
ro = lambda s: f" · readout: [italic]{escape(s)}[/italic]" if s else ""
rich_print(Panel(Text(cx["gen_src"].strip()), title=f"[red1]clean SRC (c)[/red1]{ro(cx.get('ro_src'))}", title_align="left"))
rich_print(Panel(Text(cx["gen_dst"].strip()), title=f"[sea_green3]clean DST (d)[/sea_green3]{ro(cx.get('ro_dst'))}", title_align="left"))
rich_print(f"  swept ({escape(F_NAME)}): [bold]{r0['causal']}[/bold]")

idx 15 · c = total_vowels: the total number of vowels across all words
         d = longest_word_length: the length of the longest word
task 33 tok (incl. 1 task-to-progress tok) · progress 42 tok · pad 0/2 (src/dst)

╭─ SRC · task span (light red) · progress (yellow) · pivot (salmon) · L=139 ──────────────────────────────────────╮
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of distinct words\n                                                                              │
│ (2) the number of words longer than five characters\n                                                           │
│ (3) the total number of vowels across all words\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(set(words))\n                                                                                       │
│     b = sum(1 for w in words if len(w) > 5)\n                                                                   │
│     c =                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · task span (light green) · progress (yellow) · pivot (salmon) · L=139 ────────────────────────────────────╮
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ (1) the number of distinct words\n                                                                              │
│ (2) the number of words longer than five characters\n                                                           │
│ (3) the length of the longest word\n                                                                            │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(set(words))\n                                                                                       │
│     b = sum(1 for w in words if len(w) > 5)\n                                                                   │
│     c =                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=138 ' =' => Counting characters that are vowels.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=138 ' =' => Identify the longest string length in the list.

╭─ clean SRC (c) · readout: Counting characters that are vowels. ─────────────────────────────────────────────────╮
│ sum(w.count(v) for w in words for v in 'aeiouAEIOU')                                                            │
│     return a, b, c                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ clean DST (d) · readout: Identify the longest string length in the list. ──────────────────────────────────────╮
│ max(len(w) for w in words)                                                                                      │
│     return a, b, c                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

swept (kv·goal250 · progress · V×4): src

In [ ]:
# ── the patched run on the picked case (intervention knobs) + latent sanity check ──
#------------------------------------------------------------------------------------------------------
INSPECT_SPAN = "progress"          # FILTER_SPAN | "task" | "progress" | "task+progress" | "none"
INSPECT_SCHEME = "kv·goal250"      # FILTER_SCHEME | "residual" | "kv·goal250" | "kv·all" | "none"
INSPECT_V = 4.0                  # V-boost (kv only); None -> FILTER_V
#------------------------------------------------------------------------------------------------------
I_METHOD, I_HEADS = SCHEMES[INSPECT_SCHEME]
v_run = (FILTER_V if INSPECT_V is None else INSPECT_V) if I_METHOD == "kv" else 1.0
if I_METHOD != "kv" and INSPECT_V not in (None, 1.0):
    raise ValueError(f"INSPECT_V only applies to kv (the {I_METHOD} scheme has no value boost)")
span_run = "none" if I_METHOD == "none" else INSPECT_SPAN
stack_x = make_stack(Gx, I_METHOD, span_run, I_HEADS, v_run)
run_name = cfg_name(I_METHOD, I_HEADS, span_run, v_run)

# sanity: every pre-pivot span that is NOT patched must equal the clean DST run (every layer's output); the patched
# span must equal SRC under `residual` and stay DST under `kv` (k/v is read from SRC only from the pivot on)
patched = set() if span_run == "none" else set(span_run.split("+"))
spans = dict(task=Gx["W"]["task"], progress=Gx["W"]["progress"])
all_d = sorted({q for _, d in spans.values() for q in d})
h_dst_x = PP.layer_outputs(mt, Gx["dst_tok"], all_d, LAYERS)
with stack_x():
    h_pat_x = PP.layer_outputs(mt, Gx["dst_tok"], all_d, LAYERS)
row = {q: i for i, q in enumerate(all_d)}
t = Table(title=f"latents before the pivot under {escape(run_name)} (max |Δ| over all layers)")
for c in ("span", "tokens", "patched?", "vs clean DST", "vs clean SRC", "check"):
    t.add_column(c)
for name, (s_pos, d_pos) in spans.items():
    if not d_pos:
        continue
    idx = [row[q] for q in d_pos]
    d_dst = max(float((h_pat_x[l][idx] - h_dst_x[l][idx]).abs().max()) for l in LAYERS)
    is_p = name in patched
    if is_p and I_METHOD == "residual":
        h_src_x = PP.layer_outputs(mt, Gx["src_tok"], s_pos, LAYERS)
        d_src = max(float((h_pat_x[l][idx] - h_src_x[l]).abs().max()) for l in LAYERS)
        ok = d_src < 1e-2
    else:
        d_src = None
        ok = d_dst == 0          # not patched (or kv, which leaves pre-pivot latents alone): identical to clean DST
    t.add_row(name, str(len(d_pos)), "yes" if is_p else "no", f"{d_dst:.3g}", "—" if d_src is None else f"{d_src:.3g}",
              Text("OK" if ok else "FAIL", style="green4" if ok else "bold red"))
Console(width=140).print(t)

if I_METHOD == "none":
    gx, rx = cx["gen_dst"], cx.get("ro_dst")
else:
    gx, rx = run_patched(Gx, stack_x, INSPECT_READOUT)
causal_x, readout_x = judge(Gx, cx, gx, rx)
rich_print(Panel(Text(gx.strip()), title=f"[salmon1]patched · {escape(run_name)}[/salmon1] · oracle: [bold]{causal_x['label']}[/bold]"
                 + (f" · readout→{readout_x['patched']}" if readout_x else "") + ro(rx), title_align="left"))
rich_print(f"  oracle why: {escape(causal_x['why'])}")
rich_print(f"  swept ({escape(F_NAME)}): [bold]{r0['causal']}[/bold]"
           + ("" if run_name == F_NAME else f"  ·  this run differs from the swept config ({escape(run_name)})"))

 latents before the pivot under kv·goal250 · progress · V×4 (max |Δ|  
                           over all layers)                           
┏━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┓
┃ span     ┃ tokens ┃ patched? ┃ vs clean DST ┃ vs clean SRC ┃ check ┃
┡━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━┩
│ task     │ 33     │ no       │ 0            │ —            │ OK    │
│ progress │ 42     │ yes      │ 0            │ —            │ OK    │
└──────────┴────────┴──────────┴──────────────┴──────────────┴───────┘

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=138 ' =' => I'm counting characters in a string.

╭─ patched · kv·goal250 · progress · V×4 · oracle: src · readout→neither · readout: I'm counting characters in a ─╮
│ sum(w.count(v) for w in words for v in 'aeiouAEIOU')                                                            │
│     return a, b, c                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

oracle why: The first computation counts vowels across all words using `w.count(v) for w in words for v in 
'aeiouAEIOU'`, which directly implements sub-goal A (total number of vowels).

swept (kv·goal250 · progress · V×4): src

: 